## 1. Setup

In [ ]:
!pip install -q "ultralytics==8.4.173"

In [ ]:
%%writefile /kaggle/working/cbam_module.py
import torch
import torch.nn as nn


class CBAM(nn.Module):
    """CBAM asli (Woo et al., 2018), sesuai Rumus 2.3-2.6 di proposal.

    Channel : Mc(F)  = sigmoid( MLP(AvgPool(F)) + MLP(MaxPool(F)) )   (MLP dipakai bersama, reduction ratio r)
    Spatial : Ms(F') = sigmoid( conv7x7([AvgPool_c(F'); MaxPool_c(F')]) )
    Output  : F''    = Ms(F') * (Mc(F) * F)
    """

    def __init__(self, c1, ratio=16, kernel_size=7):
        super().__init__()
        hidden = max(c1 // ratio, 1)
        self.mlp = nn.Sequential(
            nn.Conv2d(c1, hidden, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(hidden, c1, 1, bias=False)
        )
        self.spatial = nn.Conv2d(2, 1, kernel_size, padding=kernel_size // 2, bias=False)

    def forward(self, x):
        ca = torch.sigmoid(self.mlp(x.mean((2, 3), keepdim=True)) + self.mlp(x.amax((2, 3), keepdim=True)))
        x = x * ca
        sa = torch.sigmoid(self.spatial(torch.cat([x.mean(1, keepdim=True), x.amax(1, keepdim=True)], 1)))
        return x * sa

In [ ]:
import os, sys, re, glob, shutil, random
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
from PIL import Image, ImageDraw, ImageOps
from IPython.display import display

import ultralytics
from ultralytics import YOLO
from ultralytics.utils.torch_utils import init_seeds, get_num_params, get_flops
import ultralytics.nn.tasks as T

sys.path.insert(0, "/kaggle/working")
from cbam_module import CBAM
T.CBAM = CBAM

print("ultralytics", ultralytics.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")

In [ ]:
WORK = Path("/kaggle/working")
RUNS = WORK / "runs"
RESULTS = WORK / "hasil_eksperimen.csv"

SIZE = "n"
IMGSZ, EPOCHS, BATCH = 640, 100, 16
OPTIMIZER, LR0 = "AdamW", 0.001
SEEDS = [0, 1, 2]

IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}
BASE_YAML = Path(ultralytics.__file__).parent / "cfg/models/11/yolo11.yaml"

BALANCE_LEAF = True
BALANCE_TOL = 1.5
BALANCE_MAX_REPEAT = 6

TRAIN_OVERRIDES = {"fruit": {}, "leaf": {}}

RESTORE_DIR = None                 

RUN_SMOKE = True                   
RUN_EXPORT = False                 

INPUT = {
    "fruit": Path("/kaggle/input/datasets/lalalili12345456/rockmelon-fruit/DATASET DISTRIBUTION ROCK MELON"),
    "leaf": Path("/kaggle/input/datasets/lalalili12345456/rockmelon-leaf/Diseases Rock Melon.v2-v2-aug-image-level.yolov11"),
}
NAMES = {"fruit": ["fully_ripe", "unripe", "half_ripe"], "leaf": []}
TRAIN_TASKS = ("fruit", "leaf_bal" if BALANCE_LEAF else "leaf")

for k, v in INPUT.items():
    print(f"{k:5}: ada={v.exists()} | {v}")
print("tugas yang dilatih:", TRAIN_TASKS)

In [ ]:
def restore_previous(src):
    """Salin runs/ dan CSV hasil sesi lama ke folder kerja, supaya run yang sudah selesai tidak dilatih ulang."""
    if not src:
        print("RESTORE_DIR kosong: mulai dari awal.")
        return
    src = Path(src)
    RUNS.mkdir(parents=True, exist_ok=True)
    if (src / "runs").is_dir():
        shutil.copytree(src / "runs", RUNS, dirs_exist_ok=True)
    elif (src / "hasil_runs.zip").exists():
        shutil.unpack_archive(src / "hasil_runs.zip", RUNS)
    if (src / "hasil_eksperimen.csv").exists():
        shutil.copy2(src / "hasil_eksperimen.csv", RESULTS)
    print("dipulihkan dari", src, "| run dengan best.pt:", len(list(RUNS.glob("*/weights/best.pt"))))

restore_previous(RESTORE_DIR)

## 2. Dataset

In [ ]:
from PIL import ImageFont
try:
    FONT = ImageFont.load_default(size=34)
except TypeError:
    FONT = ImageFont.load_default()

def _sub(path, name):
    """Subfolder dengan nama `name` (tidak peka huruf besar/kecil), atau None."""
    if path is not None and path.is_dir():
        for c in path.iterdir():
            if c.is_dir() and c.name.lower() == name:
                return c

def locate(root, split):
    """Return (folder_gambar, folder_label). Tidak peka huruf besar/kecil dan mencari ke dalam subfolder, jadi mendukung:
       .../TRAIN/train/images + .../TRAIN/train/labels   |   .../images/train + .../labels/train   |   satu folder berisi gambar+txt."""
    aliases = {"train": ["train"], "val": ["valid", "val", "validation"], "test": ["test"]}[split]
    dirs = [d for d in [root, *root.rglob("*")] if d.is_dir()]
    for d in dirs:
        n, par = d.name.lower(), d.parent
        if n == "images" and par.name.lower() in aliases:
            return d, (_sub(par, "labels") or d)
        if n in aliases and par.name.lower() == "images":
            lb = _sub(par.parent, "labels")
            return d, ((_sub(lb, n) if lb else None) or d)
    for d in dirs:
        if d.name.lower() in aliases and any(f.suffix.lower() in IMG_EXT for f in d.iterdir() if f.is_file()):
            return d, d
    raise FileNotFoundError(f"split '{split}' tidak ditemukan di {root}. Cek struktur: !find {root} -maxdepth 4 -type d")

def read_names(root):
    for f in root.rglob("data.yaml"):
        names = yaml.safe_load(f.read_text()).get("names")
        if isinstance(names, dict):
            names = [names[k] for k in sorted(names)]
        if names:
            return list(names)
    return None

def clean_label(src, dst):
    """ID '0.0' -> 0, koordinat di-clip ke [0,1], baris rusak dibuang. Return (ok, dibuang, label_hilang)."""
    ok = bad = 0
    out = []
    if src.exists():
        for ln in src.read_text().strip().splitlines():
            t = ln.split()
            try:
                if len(t) != 5:
                    raise ValueError
                c = int(float(t[0]))
                x, y, w, h = [min(max(float(v), 0.0), 1.0) for v in t[1:]]
                if w <= 0 or h <= 0:
                    raise ValueError
                out.append(f"{c} {x:.6f} {y:.6f} {w:.6f} {h:.6f}")
                ok += 1
            except ValueError:
                bad += 1
    dst.write_text("\n".join(out) + ("\n" if out else ""))
    return ok, bad, int(not src.exists())

def prepare(task, size=IMGSZ):
    src, out = INPUT[task], WORK / f"data_{task}"
    if out.exists():
        shutil.rmtree(out)
    stats = Counter()
    for split, name in [("train", "train"), ("val", "valid"), ("test", "test")]:
        si, sl = locate(src, split)
        oi, ol = out / name / "images", out / name / "labels"
        oi.mkdir(parents=True)
        ol.mkdir(parents=True)
        for p in sorted(si.iterdir()):
            if p.suffix.lower() not in IMG_EXT:
                continue
            im = ImageOps.exif_transpose(Image.open(p)).convert("RGB")
            im.resize((size, size), Image.BILINEAR).save(oi / f"{p.stem}.jpg", quality=95)
            ok, bad, miss = clean_label(sl / f"{p.stem}.txt", ol / f"{p.stem}.txt")
            stats["baris_label_ok"] += ok
            stats["baris_dibuang"] += bad
            stats["file_label_hilang"] += miss
    names = read_names(src) or NAMES[task]
    NAMES[task] = names
    cfg = {"path": str(out), "train": "train/images", "val": "valid/images", "test": "test/images",
           "nc": len(names), "names": {i: n for i, n in enumerate(names)}}
    (WORK / f"{task}.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
    print(task, dict(stats), "| kelas:", names)
    return names

def count_table(task):
    out, names = WORK / f"data_{task}", NAMES[task]
    rows = []
    for split in ["train", "valid", "test"]:
        cnt = Counter(int(l.split()[0]) for f in (out / split / "labels").glob("*.txt") for l in f.read_text().splitlines())
        ids = sorted(set(cnt) | set(range(len(names))))
        rows.append({"split": split, "citra": len(list((out / split / "images").glob("*.jpg"))), "objek": sum(cnt.values()),
                     **{(names[i] if i < len(names) else f"id{i}"): cnt.get(i, 0) for i in ids}})
    return pd.DataFrame(rows)

def preview(task, split="train", n=8, cols=4, seed=0):
    """Gambar kotak + 'ID:nama' -> pastikan nama kelas cocok dengan isi gambar (mis. hijau mulus = unripe)."""
    out, names = WORK / f"data_{task}", NAMES[task]
    imgs = sorted((out / split / "images").glob("*.jpg"))
    random.Random(seed).shuffle(imgs)
    tiles = []
    for p in imgs[:n]:
        im = Image.open(p).convert("RGB")
        d, (W, H) = ImageDraw.Draw(im), im.size
        for l in (out / split / "labels" / f"{p.stem}.txt").read_text().splitlines():
            c, x, y, w, h = int(l.split()[0]), *map(float, l.split()[1:])
            box = [(x - w / 2) * W, (y - h / 2) * H, (x + w / 2) * W, (y + h / 2) * H]
            d.rectangle(box, outline=(255, 0, 0), width=3)
            d.text((box[0] + 4, box[1] + 4), f"{c}:{names[c] if c < len(names) else '?'}", font=FONT, fill=(255, 255, 0), stroke_width=2, stroke_fill=(0, 0, 0))
        tiles.append(im.resize((320, 320)))
    sheet = Image.new("RGB", (320 * cols, 320 * ((len(tiles) + cols - 1) // cols)))
    for k, t in enumerate(tiles):
        sheet.paste(t, (320 * (k % cols), 320 * (k // cols)))
    return sheet

def leakage_report(task):
    """Indikasi kasar kebocoran: jumlah citra per sumber (tanggal pengambilan / prefix nama file) di tiap split."""
    out, rows = WORK / f"data_{task}", []
    for split in ["train", "valid", "test"]:
        for p in (out / split / "images").glob("*.jpg"):
            base = re.split(r"_(?:jpg|jpeg|png)\.rf\.", p.stem)[0]
            m = re.search(r"(?<!\d)(20\d{6})(?!\d)", base)
            rows.append({"split": split, "sumber": m.group(1) if m else re.sub(r"[-_]?\d+$", "", base)})
    df = pd.DataFrame(rows)
    return pd.crosstab(df["sumber"], df["split"])

### 2a. Dataset buah (kematangan)

In [ ]:
prepare("fruit")
display(count_table("fruit"))

In [ ]:
display(preview("fruit", "train"))
display(preview("fruit", "test"))

In [ ]:
cfg = yaml.safe_load((WORK / "fruit.yaml").read_text())
cfg["nc"], cfg["names"] = len(NAMES["fruit"]), {i: n for i, n in enumerate(NAMES["fruit"])}
(WORK / "fruit.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg["names"])

### 2b. Dataset daun (penyakit)

In [ ]:
prepare("leaf")
display(count_table("leaf"))
display(leakage_report("leaf"))

In [ ]:
display(preview("leaf", "train"))
display(preview("leaf", "test"))

### 2c. Seimbangkan data latih daun

In [ ]:
def balance_train(src_task="leaf", dst_task="leaf_bal", tol=1.5, max_repeat=6, seed=0):
    rnd = random.Random(seed)
    src, dst = WORK / f"data_{src_task}", WORK / f"data_{dst_task}"
    names = NAMES[src_task]
    K = len(names)

    imgs = {lf.stem: Counter(int(l.split()[0]) for l in lf.read_text().splitlines() if l.strip())
            for lf in sorted((src / "train" / "labels").glob("*.txt"))}
    rep = {s: 1 for s in imgs}
    tot = Counter()
    for s, c in imgs.items():
        tot.update(c)
    before = Counter(tot)

    def bump(s, d):
        rep[s] += d
        for k, v in imgs[s].items():
            tot[k] += d * v

    target = float(np.median([before[k] for k in range(K) if before[k] > 0]))
    lo, hi = target / tol, target * tol

    for k in sorted(range(K), key=lambda k: before[k]):
        cand = [s for s, c in imgs.items() if c[k] > 0]
        cand.sort(key=lambda s: -(imgs[s][k] / sum(imgs[s].values())))
        i = 0
        while cand and tot[k] < lo and any(rep[s] < max_repeat for s in cand):
            s = cand[i % len(cand)]
            if rep[s] < max_repeat:
                bump(s, 1)
            i += 1

    pool = [s for s, c in imgs.items() if c]
    rnd.shuffle(pool)
    def excess(s):
        return sum(v for k, v in imgs[s].items() if tot[k] > hi) / sum(imgs[s].values())
    changed = True
    while changed:
        changed = False
        for s in sorted((s for s in pool if rep[s] > 0), key=lambda s: -excess(s)):
            c = imgs[s]
            if any(tot[k] > hi for k in c) and all(tot[k] - rep[s] * v >= lo for k, v in c.items()):
                bump(s, -rep[s])
                changed = True
                break

    if dst.exists():
        shutil.rmtree(dst)
    for sp in ("valid", "test"):
        shutil.copytree(src / sp, dst / sp)
    (dst / "train" / "images").mkdir(parents=True)
    (dst / "train" / "labels").mkdir(parents=True)
    for s, r in rep.items():
        for j in range(r):
            n = s if j == 0 else f"{s}_dup{j}"
            shutil.copy2(src / "train" / "images" / f"{s}.jpg", dst / "train" / "images" / f"{n}.jpg")
            shutil.copy2(src / "train" / "labels" / f"{s}.txt", dst / "train" / "labels" / f"{n}.txt")

    NAMES[dst_task] = names
    cfg = {"path": str(dst), "train": "train/images", "val": "valid/images", "test": "test/images",
           "nc": K, "names": {i: n for i, n in enumerate(names)}}
    (WORK / f"{dst_task}.yaml").write_text(yaml.safe_dump(cfg, sort_keys=False))

    rep_df = pd.DataFrame({"sebelum": [before[k] for k in range(K)], "sesudah": [tot[k] for k in range(K)]}, index=names)
    nz = rep_df[rep_df > 0].dropna()
    print(f"target ~{target:.0f} objek/kelas, rentang [{lo:.0f}, {hi:.0f}] | rasio terbesar:terkecil "
          f"{before.most_common(1)[0][1] / max(min(before[k] for k in range(K)), 1):.1f}:1 -> "
          f"{max(tot.values()) / max(min(tot[k] for k in range(K)), 1):.1f}:1")
    print("citra train:", len(imgs), "->", sum(rep.values()), "| citra unik dibuang:", sum(r == 0 for r in rep.values()),
          "| citra diulang:", sum(r > 1 for r in rep.values()))
    kurang = [names[k] for k in range(K) if tot[k] < lo]
    if kurang:
        print("PERHATIAN: belum mencapai batas bawah (max_repeat tercapai):", kurang, "-> naikkan max_repeat atau tol, atau terima apa adanya")
    return rep_df

In [ ]:
if BALANCE_LEAF:
    bal_report = balance_train(tol=BALANCE_TOL, max_repeat=BALANCE_MAX_REPEAT)
    display(bal_report)
    display(count_table("leaf_bal"))
else:
    print("BALANCE_LEAF=False: data daun dipakai apa adanya (task 'leaf').")

## 3. Model: YOLO11n + CBAM

In [ ]:
def _div8(x):
    return int(np.ceil(x / 8) * 8)

def make_cbam_yaml(size, nc):
    base = yaml.safe_load(BASE_YAML.read_text())
    _, width, max_ch = base["scales"][size]
    *body, det = base["backbone"] + base["head"]
    feed = list(det[0])
    old2new, cbam_at, new = {}, {}, []

    def remap(f):
        return [remap(x) for x in f] if isinstance(f, list) else (f if f < 0 else old2new[f])

    for i, (f, n, m, a) in enumerate(body):
        new.append([remap(f), n, m, a])
        old2new[i] = len(new) - 1
        if i in feed:
            new.append([-1, 1, "CBAM", [_div8(min(a[0], max_ch) * width)]])
            cbam_at[i] = len(new) - 1
    new.append([[cbam_at[i] for i in feed], det[1], det[2], det[3]])
    old2new[len(body)] = len(new) - 1
    nb = len(base["backbone"])
    base["backbone"], base["head"], base["nc"] = new[:nb], new[nb:], nc
    path = WORK / f"yolo11{size}-cbam.yaml"
    path.write_text(yaml.safe_dump(base, sort_keys=False, default_flow_style=None))
    return path, old2new

def build_cbam(size, nc, seed):
    init_seeds(seed)
    path, old2new = make_cbam_yaml(size, nc)
    model = YOLO(str(path))
    donor = YOLO(f"yolo11{size}.pt").model.state_dict()
    sd, moved = model.model.state_dict(), 0
    for k, v in donor.items():
        p = k.split(".")
        if p[0] != "model" or not p[1].isdigit() or int(p[1]) not in old2new:
            continue
        p[1] = str(old2new[int(p[1])])
        nk = ".".join(p)
        if nk in sd and sd[nk].shape == v.shape:
            sd[nk] = v.clone()
            moved += 1
    model.model.load_state_dict(sd)
    model.load(model.model)
    print(f"[CBAM] {moved}/{len(sd)} tensor diisi dari yolo11{size}.pt; sisanya = modul CBAM + head klasifikasi")
    return model

def complexity(m):
    return get_num_params(m.model) / 1e6, get_flops(m.model, IMGSZ)

In [ ]:
nc = len(NAMES["fruit"])
cb = build_cbam(SIZE, nc, seed=0)
print([(i, type(m).__name__) for i, m in enumerate(cb.model.model) if type(m).__name__ in ("CBAM", "Detect")])
extra = sum(p.numel() for m in cb.model.modules() if isinstance(m, CBAM) for p in m.parameters())
total = get_num_params(cb.model)
print(f"Parameter tambahan dari CBAM: {extra:,} ({100 * extra / total:.2f}% dari {total:,})")
cb.model.eval()
with torch.no_grad():
    out = cb.model(torch.zeros(1, 3, IMGSZ, IMGSZ).to(next(cb.model.parameters()).device))
print("forward OK")
print(open(WORK / f"yolo11{SIZE}-cbam.yaml").read()[-900:])

## 4. Training & evaluasi

In [ ]:
def run_state(name):
    """'selesai' | 'sebagian' | 'baru'. Setelah training selesai Ultralytics menyimpan last.pt dengan epoch = -1."""
    last = RUNS / name / "weights" / "last.pt"
    if not last.exists():
        return "baru"
    try:
        ep = torch.load(last, map_location="cpu", weights_only=False).get("epoch", -1)
    except Exception as e:
        print("  (peringatan) last.pt tidak terbaca, run dimulai ulang:", e)
        return "baru"
    return "selesai" if ep == -1 and (RUNS / name / "weights" / "best.pt").exists() else "sebagian"

def train_one(task, variant, seed, epochs=EPOCHS):
    name = f"{task}_{variant}_s{seed}"
    state = run_state(name)
    if state == "selesai":
        print("lewati (sudah selesai):", name)
        return
    if state == "sebagian":
        print("lanjutkan (resume):", name)
        YOLO(str(RUNS / name / "weights" / "last.pt")).train(resume=True)
        return
    nc = len(NAMES[task])
    model = YOLO(f"yolo11{SIZE}.pt") if variant == "baseline" else build_cbam(SIZE, nc, seed)
    extra = TRAIN_OVERRIDES.get(task.split("_")[0], {})
    model.train(data=str(WORK / f"{task}.yaml"), imgsz=IMGSZ, epochs=epochs, batch=BATCH,
                optimizer=OPTIMIZER, lr0=LR0, seed=seed, deterministic=True, device=0, workers=2,
                project=str(RUNS), name=name, exist_ok=True, plots=True, verbose=False, **extra)

def evaluate(task, variant, seed, split="test"):
    name = f"{task}_{variant}_s{seed}"
    best = RUNS / name / "weights" / "best.pt"
    n_p, gf = complexity(YOLO(str(best)))
    met = YOLO(str(best)).val(data=str(WORK / f"{task}.yaml"), split=split, imgsz=IMGSZ, batch=BATCH, device=0,
                              project=str(RUNS), name=f"{name}_{split}", exist_ok=True, plots=True, verbose=False)
    b, names = met.box, NAMES[task]
    P, R = float(b.mp), float(b.mr)
    row = dict(task=task, variant=variant, seed=seed, split=split, precision=P, recall=R,
               f1=2 * P * R / (P + R + 1e-9), map50=float(b.map50), map50_95=float(b.map),
               params_M=n_p, gflops=gf, pt_MB=best.stat().st_size / 1e6)
    for i, c in enumerate(b.ap_class_index):
        row[f"AP50_{names[c]}"] = float(b.ap50[i])
        row[f"AP50-95_{names[c]}"] = float(b.ap[i])
        row[f"P_{names[c]}"], row[f"R_{names[c]}"] = float(b.p[i]), float(b.r[i])
    return row

def run_all(tasks=TRAIN_TASKS, seeds=SEEDS, epochs=EPOCHS):
    rows = pd.read_csv(RESULTS).to_dict("records") if RESULTS.exists() else []
    done = {(r["task"], r["variant"], r["seed"], r["split"]) for r in rows}
    for seed in seeds:
        for task in tasks:
            for variant in ["baseline", "cbam"]:
                if (task, variant, seed, "test") in done:
                    continue
                train_one(task, variant, seed, epochs)
                rows = [r for r in rows if (r["task"], r["variant"], r["seed"]) != (task, variant, seed)]
                rows += [evaluate(task, variant, seed, "test"), evaluate(task, variant, seed, "val")]
                pd.DataFrame(rows).to_csv(RESULTS, index=False)
    return pd.DataFrame(rows)

def smoke(task):
    nc = len(NAMES[task])
    for variant in ["baseline", "cbam"]:
        m = YOLO(f"yolo11{SIZE}.pt") if variant == "baseline" else build_cbam(SIZE, nc, 0)
        m.train(data=str(WORK / f"{task}.yaml"), imgsz=IMGSZ, epochs=1, batch=BATCH, optimizer=OPTIMIZER, lr0=LR0,
                device=0, workers=2, project=str(WORK / "smoke"), name=variant, exist_ok=True, plots=False, verbose=False)
    print("smoke test OK:", task)

In [ ]:
if RUN_SMOKE:
    for t in TRAIN_TASKS:
        smoke(t)

### Training penuh

### Cara pakai

In [ ]:
import gc, time, logging, contextlib
from IPython.display import clear_output

LOGS = WORK / "logs"
LOGS.mkdir(exist_ok=True)
T_FRUIT, T_LEAF = TRAIN_TASKS

def _fmt(sec):
    m, s = divmod(int(sec), 60); h, m = divmod(m, 60)
    return f"{h}j {m:02d}m" if h else f"{m}m {s:02d}s"

@contextlib.contextmanager
def senyap(logfile):
    """Arahkan semua output training/val ke file log, supaya layar notebook tidak menumpuk."""
    lg = logging.getLogger("ultralytics"); old = lg.level
    lg.setLevel(logging.WARNING)
    try:
        with open(logfile, "a") as f, contextlib.redirect_stdout(f), contextlib.redirect_stderr(f):
            yield
    finally:
        lg.setLevel(old)

def pasang_progres(model, name, layar):
    """Callback: tiap epoch hapus output lama, lalu tampilkan satu blok ringkas."""
    st = dict(t0=time.time(), ep0=None, best=0.0)
    def _cb(trainer):
        ep, tot = trainer.epoch + 1, trainer.epochs
        if st["ep0"] is None:
            st["ep0"] = ep - 1
        m = trainer.metrics
        mp50  = float(m.get("metrics/mAP50(B)", float("nan")))
        mp95  = float(m.get("metrics/mAP50-95(B)", float("nan")))
        P, R  = float(m.get("metrics/precision(B)", float("nan"))), float(m.get("metrics/recall(B)", float("nan")))
        if mp95 == mp95: st["best"] = max(st["best"], mp95)
        try: box, cls, dfl = [float(x) for x in trainer.tloss]
        except Exception: box = cls = dfl = float("nan")
        el = time.time() - st["t0"]; done = ep - st["ep0"]
        eta = el / max(done, 1) * (tot - ep)
        clear_output(wait=True)
        print(f"{name}\n"
              f"  epoch {ep}/{tot}  |  lewat {_fmt(el)}  |  sisa ~{_fmt(eta)}\n"
              f"  loss  box {box:.3f}  cls {cls:.3f}  dfl {dfl:.3f}\n"
              f"  val   P {P:.3f}  R {R:.3f}  mAP50 {mp50:.3f}  mAP50-95 {mp95:.3f}  (terbaik {st['best']:.3f})",
              file=layar)
    model.add_callback("on_fit_epoch_end", _cb)

def _latih(task, variant, seed, epochs, name, layar):
    state = run_state(name)
    if state == "sebagian":
        m = YOLO(str(RUNS / name / "weights" / "last.pt"))
        pasang_progres(m, name, layar)
        m.train(resume=True)
        return
    nc = len(NAMES[task])
    m = YOLO(f"yolo11{SIZE}.pt") if variant == "baseline" else build_cbam(SIZE, nc, seed)
    pasang_progres(m, name, layar)
    extra = TRAIN_OVERRIDES.get(task.split("_")[0], {})
    m.train(data=str(WORK / f"{task}.yaml"), imgsz=IMGSZ, epochs=epochs, batch=BATCH,
            optimizer=OPTIMIZER, lr0=LR0, seed=seed, deterministic=True, device=0, workers=2,
            project=str(RUNS), name=name, exist_ok=True, plots=True, verbose=False, **extra)

def run_satu(task, variant, seed, epochs=EPOCHS):
    """Latih + evaluasi SATU run, simpan ke CSV. Aman dijalankan ulang: yang sudah selesai dilewati,
    yang terputus dilanjutkan dari last.pt."""
    name = f"{task}_{variant}_s{seed}"
    rows = pd.read_csv(RESULTS).to_dict("records") if RESULTS.exists() else []
    if any((r["task"], r["variant"], r["seed"], r["split"]) == (task, variant, seed, "test") for r in rows):
        print("sudah selesai, dilewati:", name)
        return
    layar, t0 = sys.stdout, time.time()
    try:
        with senyap(LOGS / f"{name}.log"):
            if run_state(name) != "selesai":
                _latih(task, variant, seed, epochs, name, layar)
            baru = [evaluate(task, variant, seed, "test"), evaluate(task, variant, seed, "val")]
    finally:
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    rows = [r for r in rows if (r["task"], r["variant"], r["seed"]) != (task, variant, seed)] + baru
    pd.DataFrame(rows).to_csv(RESULTS, index=False)
    clear_output(wait=True)
    t = baru[0]
    print(f"SELESAI {name}  ({_fmt(time.time() - t0)})\n"
          f"  test  P {t['precision']:.3f}  R {t['recall']:.3f}  F1 {t['f1']:.3f}  "
          f"mAP50 {t['map50']:.3f}  mAP50-95 {t['map50_95']:.3f}")

def status_run():
    """Tabel 12 run: selesai / sebagian / baru."""
    sel = set()
    if RESULTS.exists():
        d = pd.read_csv(RESULTS)
        sel = {(r.task, r.variant, r.seed) for r in d[d.split == "test"].itertuples()}
    out = []
    for seed in SEEDS:
        for task in TRAIN_TASKS:
            for v in ("baseline", "cbam"):
                out.append(dict(run=f"{task}_{v}_s{seed}",
                                status="selesai" if (task, v, seed) in sel else run_state(f"{task}_{v}_s{seed}")))
    return pd.DataFrame(out)

status_run()

#### Seed 0

In [ ]:
run_satu(T_FRUIT, "baseline", 0)

In [ ]:
run_satu(T_FRUIT, "cbam", 0)

In [ ]:
run_satu(T_LEAF, "baseline", 0)

In [ ]:
run_satu(T_LEAF, "cbam", 0)

#### Seed 1

In [ ]:
run_satu(T_FRUIT, "baseline", 1)

In [ ]:
run_satu(T_FRUIT, "cbam", 1)

In [ ]:
run_satu(T_LEAF, "baseline", 1)

In [ ]:
run_satu(T_LEAF, "cbam", 1)

#### Seed 2

In [ ]:
run_satu(T_FRUIT, "baseline", 2)

In [ ]:
run_satu(T_FRUIT, "cbam", 2)

In [ ]:
run_satu(T_LEAF, "baseline", 2)

In [ ]:
run_satu(T_LEAF, "cbam", 2)

#### Tabel hasil (jalankan setelah semua run selesai, atau kapan saja untuk melihat progres)

In [ ]:
display(status_run())
df = pd.read_csv(RESULTS)
df[df.split == "test"][["task", "variant", "seed", "precision", "recall", "f1", "map50", "map50_95"]].round(4)

## 5. Ringkasan hasil

In [ ]:
df = pd.read_csv(RESULTS)
cols = ["precision", "recall", "f1", "map50", "map50_95", "params_M", "gflops", "pt_MB"]

def summarize(split="test"):
    return df[df.split == split].groupby(["task", "variant"])[cols].agg(["mean", "std"]).round(4)

def paired(task, split="test", metric="map50_95"):
    d = df[(df.task == task) & (df.split == split)].pivot(index="seed", columns="variant", values=metric)
    d["selisih (cbam - baseline)"] = d["cbam"] - d["baseline"]
    return d.round(4)

def per_class(task, split="test"):
    d = df[(df.task == task) & (df.split == split)]
    cs = [c for c in d.columns if c.startswith("AP50-95_") and d[c].notna().any()]
    return d.groupby("variant")[cs].agg(["mean", "std"]).round(4).T if cs else pd.DataFrame()

def kriteria_a(task, metric="map50_95"):
    d = paired(task, metric=metric)
    diff, std_base = d["selisih (cbam - baseline)"], d["baseline"].std()
    ok = bool((diff > 0).all() and diff.mean() > std_base)
    print(f"[{task}] selisih per seed {diff.tolist()} | rata-rata {diff.mean():+.4f} | std baseline {std_base:.4f} "
          f"-> kriteria (a) {'TERPENUHI' if ok else 'TIDAK terpenuhi'}")

display(summarize("test"))
for t in sorted(df.task.unique()):
    if not {"baseline", "cbam"} <= set(df[(df.task == t) & (df.split == "test")].variant):
        print("\n==", t, "-> belum lengkap (baseline & cbam), dilewati")
        continue
    print("\n==", t)
    display(paired(t))
    kriteria_a(t)
    display(per_class(t))
print("\nKriteria (b) (tambahan waktu inferensi & penurunan FPS <= 10%) diukur di perangkat Android, bukan di notebook ini.")

## 5b. Estimasi kapan buah matang

In [ ]:
STAGE_DAYS = {"unripe": (5, 8), "half_ripe": (2, 4), "fully_ripe": (0, 0)}
PRED_MODEL = RUNS / "fruit_cbam_s0" / "weights" / "best.pt"
PRED_CONF = 0.25

from datetime import date, timedelta

def _load640(path):
    return ImageOps.exif_transpose(Image.open(path)).convert("RGB").resize((IMGSZ, IMGSZ), Image.BILINEAR)

def estimasi_matang(path, model_path=PRED_MODEL, conf=PRED_CONF, hari_ini=None, tampil=True):
    hari_ini = hari_ini or date.today()
    model = YOLO(str(model_path))
    im = _load640(path)
    r = model.predict(im, imgsz=IMGSZ, conf=conf, device=0 if torch.cuda.is_available() else "cpu", verbose=False)[0]
    d, rows = ImageDraw.Draw(im), []
    for i, b in enumerate(r.boxes):
        nm = model.names[int(b.cls)]
        lo, hi = STAGE_DAYS.get(nm, (None, None))
        x1, y1, x2, y2 = b.xyxy[0].tolist()
        label = nm if lo is None else ("siap panen" if hi == 0 else f"{nm}: {lo}-{hi} hari")
        d.rectangle([x1, y1, x2, y2], outline=(255, 0, 0), width=3)
        d.text((x1 + 4, y1 + 4), label, font=FONT, fill=(255, 255, 0), stroke_width=2, stroke_fill=(0, 0, 0))
        rows.append(dict(buah=i + 1, tahap=nm, conf=round(float(b.conf[0]), 3), sisa_hari_min=lo, sisa_hari_maks=hi,
                         matang_paling_cepat=None if lo is None else hari_ini + timedelta(days=lo),
                         matang_paling_lambat=None if hi is None else hari_ini + timedelta(days=hi)))
    if tampil:
        display(im.resize((480, 480)))
    return pd.DataFrame(rows)

for _p in sorted((WORK / "data_fruit" / "test" / "images").glob("*.jpg"))[:3]:
    print(_p.name)
    display(estimasi_matang(_p))

In [ ]:
PRED_CSV = None

def fitur_citra(path, det):
    im = _load640(path)
    r = det.predict(im, imgsz=IMGSZ, conf=PRED_CONF, device=0 if torch.cuda.is_available() else "cpu", verbose=False)[0]
    if len(r.boxes) == 0:
        return None
    i = int(r.boxes.conf.argmax())
    x1, y1, x2, y2 = [int(v) for v in r.boxes.xyxy[i].tolist()]
    hsv = np.asarray(im.crop((x1, y1, max(x2, x1 + 1), max(y2, y1 + 1))).convert("HSV"), dtype=float) / 255
    f = dict(conf=float(r.boxes.conf[i]), luas=(x2 - x1) * (y2 - y1) / IMGSZ ** 2,
             H=hsv[..., 0].mean(), S=hsv[..., 1].mean(), V=hsv[..., 2].mean())
    for k in sorted(det.names):
        f[f"kelas_{det.names[k]}"] = int(int(r.boxes.cls[i]) == k)
    return f

if PRED_CSV:
    from sklearn.ensemble import RandomForestRegressor
    from sklearn.dummy import DummyRegressor
    from sklearn.model_selection import GroupKFold, cross_val_predict
    from sklearn.metrics import mean_absolute_error

    meta = pd.read_csv(PRED_CSV, parse_dates=["tanggal_foto", "tanggal_matang"])
    meta["sisa_hari"] = (meta.tanggal_matang - meta.tanggal_foto).dt.days
    det = YOLO(str(PRED_MODEL))
    fs = [fitur_citra(p, det) for p in meta.file]
    ok = np.array([f is not None for f in fs])
    print(f"citra dipakai: {ok.sum()}/{len(ok)} (sisanya tidak terdeteksi buah)")
    X = pd.DataFrame([f for f in fs if f is not None])
    y, g = meta.loc[ok, "sisa_hari"].values, meta.loc[ok, "id_buah"].values
    assert len(set(g)) >= 3, "butuh minimal 3 buah berbeda untuk validasi per buah"
    cv = GroupKFold(n_splits=min(5, len(set(g))))
    rf = RandomForestRegressor(n_estimators=300, min_samples_leaf=2, random_state=0)
    mae_rf = mean_absolute_error(y, cross_val_predict(rf, X, y, groups=g, cv=cv))
    mae_0 = mean_absolute_error(y, cross_val_predict(DummyRegressor(strategy="mean"), X, y, groups=g, cv=cv))
    print(f"MAE validasi per-buah: model {mae_rf:.2f} hari | tebakan rata-rata {mae_0:.2f} hari ({len(set(g))} buah)")
    rf.fit(X, y)

    def prediksi_sisa_hari(path):
        f = fitur_citra(path, det)
        return None if f is None else float(rf.predict(pd.DataFrame([f])[X.columns])[0])
else:
    print("PRED_CSV kosong: bagian regresi dilewati (pakai 5b-1 saja).")

## 5c. Keputusan kematangan dari warna DAN tekstur (dapat dibaca) + pengatur cahaya

In [ ]:
import cv2
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

MODE_CAHAYA = ["none", "clahe", "clahe+wb"]
GANGGUAN = [None, "gelap", "terang", "hangat", "dingin"]

def normalisasi_cahaya(rgb, mode="clahe"):
    """rgb uint8 HxWx3. 'clahe' meratakan kecerahan (kanal L); '+wb' menambah keseimbangan putih gray-world
    yang diestimasi dari SELURUH citra (bisa merusak warna kalau adegan didominasi satu warna, makanya diuji)."""
    if mode == "none":
        return rgb
    x = rgb
    if "wb" in mode:
        m = rgb.reshape(-1, 3).astype(np.float32).mean(0)
        x = np.clip(rgb.astype(np.float32) * (m.mean() / (m + 1e-6)), 0, 255).astype(np.uint8)
    lab = cv2.cvtColor(x, cv2.COLOR_RGB2LAB)
    lab[..., 0] = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8)).apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

def terapkan_gangguan(rgb, g):
    if g is None:
        return rgb
    x = rgb.astype(np.float32) / 255
    if g == "gelap":    x = x ** 1.8 * 0.7
    elif g == "terang": x = np.clip(x * 1.5 + 0.05, 0, 1)
    elif g == "hangat": x = np.clip(x * np.array([1.2, 1.0, 0.8], np.float32), 0, 1)
    elif g == "dingin": x = np.clip(x * np.array([0.8, 1.0, 1.2], np.float32), 0, 1)
    return (x * 255).astype(np.uint8)

def _lbp_ciri(g):
    c = g[1:-1, 1:-1]
    code = np.zeros(c.shape, np.uint8)
    for k, (dy, dx) in enumerate([(-1, -1), (-1, 0), (-1, 1), (0, 1), (1, 1), (1, 0), (1, -1), (0, -1)]):
        nb = g[1 + dy:g.shape[0] - 1 + dy, 1 + dx:g.shape[1] - 1 + dx]
        code |= ((nb >= c).astype(np.uint8) << k)
    bits = np.unpackbits(code[..., None], axis=-1)
    trans = (bits != np.roll(bits, 1, axis=-1)).sum(-1)
    return float((trans <= 2).mean()), float((trans >= 6).mean())

def fitur_crop(rgb):
    """Satu baris fitur dari potongan buah. Awalan w_ = warna, t_ = tekstur (jaring kulit)."""
    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    lab = cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB).astype(np.float32)
    H, S, V = [hsv[..., k].astype(np.float32) for k in range(3)]
    ok, n = (S > 40) & (V > 40), H.size
    Hm = H[ok] if ok.any() else H.ravel()
    f = {"w_H": Hm.mean(), "w_Hstd": Hm.std(), "w_S": S.mean() / 255, "w_V": V.mean() / 255,
         "w_a": lab[..., 1].mean() - 128, "w_b": lab[..., 2].mean() - 128,
         "w_hijau": (ok & (H >= 35) & (H <= 85)).sum() / n,
         "w_kuning": (ok & (H >= 20) & (H < 35)).sum() / n,
         "w_oranye": (ok & (H >= 8) & (H < 20)).sum() / n}
    g = cv2.cvtColor(cv2.resize(rgb, (96, 96), interpolation=cv2.INTER_AREA), cv2.COLOR_RGB2GRAY)
    gb = cv2.GaussianBlur(g, (3, 3), 0)
    med = float(np.median(gb))
    edges = cv2.Canny(gb, int(max(0, 0.66 * med)), int(min(255, 1.33 * med)))
    gx, gy = cv2.Sobel(gb, cv2.CV_32F, 1, 0), cv2.Sobel(gb, cv2.CV_32F, 0, 1)
    mu, sd = float(g.mean()) + 1e-6, float(g.std()) + 1e-6
    f["t_tepi"] = float((edges > 0).mean())
    f["t_lap"] = float(cv2.Laplacian(gb, cv2.CV_32F).var()) / (sd ** 2)
    f["t_grad"] = float(np.sqrt(gx ** 2 + gy ** 2).mean()) / mu
    f["t_kontras"] = sd / mu
    f["t_lbp_seragam"], f["t_lbp_ramai"] = _lbp_ciri(gb)
    return f

_CACHE = {}
def fitur_split(split, mode="clahe", gangguan=None):
    """Fitur dari kotak ANOTASI (ground truth) di tiap split; split = train | valid | test."""
    key = (split, mode, gangguan)
    if key in _CACHE:
        return _CACHE[key]
    d, rows = WORK / "data_fruit" / split, []
    for p in sorted((d / "images").glob("*.jpg")):
        lines = [l.split() for l in (d / "labels" / f"{p.stem}.txt").read_text().splitlines() if l.strip()]
        if not lines:
            continue
        im = normalisasi_cahaya(terapkan_gangguan(np.asarray(Image.open(p).convert("RGB")), gangguan), mode)
        Hh, Ww = im.shape[:2]
        for c, x, y, w, h in lines:
            x, y, w, h = map(float, (x, y, w, h))
            x1, y1 = max(int((x - w / 2) * Ww), 0), max(int((y - h / 2) * Hh), 0)
            x2, y2 = min(int((x + w / 2) * Ww), Ww), min(int((y + h / 2) * Hh), Hh)
            if x2 - x1 < 8 or y2 - y1 < 8:
                continue
            rows.append({**fitur_crop(im[y1:y2, x1:x2]), "kelas": int(float(c)), "m_file": p.name, "m_box": (x1, y1, x2, y2)})
    _CACHE[key] = pd.DataFrame(rows)
    return _CACHE[key]

def latih_pohon(tr, va, kol, depths=(2, 3, 4, 5)):
    """Decision tree dangkal; kedalaman dipilih dari data VALID (bukan test)."""
    best = None
    for d in depths:
        clf = DecisionTreeClassifier(max_depth=d, min_samples_leaf=5, class_weight="balanced", random_state=0).fit(tr[kol], tr["kelas"])
        f = f1_score(va["kelas"], clf.predict(va[kol]), average="macro")
        if best is None or f > best[0] + 1e-9:
            best = (f, d, clf)
    return best[2], best[1]

print("fitur siap. Kelas buah:", NAMES["fruit"])

In [ ]:
_kol = fitur_split("train", "none").columns
W = [c for c in _kol if c.startswith("w_")]
T = [c for c in _kol if c.startswith("t_")]
KOLOM = {"warna": W, "tekstur": T, "warna+tekstur": W + T}

hasil, simpan = [], {}
for mode in MODE_CAHAYA:
    tr, va = fitur_split("train", mode), fitur_split("valid", mode)
    for nama, kol in KOLOM.items():
        clf, dalam = latih_pohon(tr, va, kol)
        simpan[(mode, nama)] = (clf, kol)
        for split in ("valid", "test"):
            for g in GANGGUAN:
                d = fitur_split(split, mode, g)
                pr = clf.predict(d[kol])
                hasil.append(dict(split=split, cahaya=mode, fitur=nama, gangguan=g or "asli", depth=dalam,
                                  acc=accuracy_score(d["kelas"], pr), f1=f1_score(d["kelas"], pr, average="macro")))
hasil = pd.DataFrame(hasil)

urut = ["asli"] + [g for g in GANGGUAN if g]
print("AKURASI TEST (baris: fitur x mode cahaya; kolom: kondisi cahaya uji)")
display(hasil[hasil.split == "test"].pivot_table(index=["fitur", "cahaya"], columns="gangguan", values="acc")[urut].round(3))

pilih = hasil[(hasil.split == "valid") & (hasil.fitur == "warna+tekstur")].groupby("cahaya").acc.mean()
MODE_TERPILIH = pilih.idxmax()
print("\nrata-rata akurasi VALID (warna+tekstur) per mode cahaya:", pilih.round(3).to_dict(), "->", MODE_TERPILIH)

clf_final, kol_final = simpan[(MODE_TERPILIH, "warna+tekstur")]
te = fitur_split("test", MODE_TERPILIH, None)
cm = confusion_matrix(te["kelas"], clf_final.predict(te[kol_final]), labels=range(len(NAMES["fruit"])))
print("\nConfusion matrix TEST (baris = sebenarnya, kolom = prediksi):")
display(pd.DataFrame(cm, index=NAMES["fruit"], columns=NAMES["fruit"]))
print("Aturan pohon (bisa dibaca & dilaporkan):")
print(export_text(clf_final, feature_names=kol_final))

In [ ]:
def _iou(a, b):
    ix = max(0.0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0.0, min(a[3], b[3]) - max(a[1], b[1]))
    inter = ix * iy
    return inter / ((a[2] - a[0]) * (a[3] - a[1]) + (b[2] - b[0]) * (b[3] - b[1]) - inter + 1e-9)

def banding_yolo_vs_pohon(model_path=None, iou_min=0.5):
    det = YOLO(str(model_path or PRED_MODEL))
    d = fitur_split("test", MODE_TERPILIH, None).copy()
    d["pohon"] = clf_final.predict(d[kol_final])
    folder, preds = WORK / "data_fruit" / "test" / "images", {}
    files = [str(folder / f) for f in d["m_file"].unique()]
    for r in det.predict(files, imgsz=IMGSZ, conf=PRED_CONF, device=0 if torch.cuda.is_available() else "cpu", verbose=False, stream=True):
        preds[Path(r.path).name] = (r.boxes.xyxy.cpu().numpy(), r.boxes.cls.cpu().numpy().astype(int))
    yolo = []
    for f, box in zip(d["m_file"], d["m_box"]):
        bx, cl = preds.get(f, (np.zeros((0, 4)), np.zeros(0, int)))
        ious = [_iou(box, b) for b in bx]
        yolo.append(int(cl[int(np.argmax(ious))]) if ious and max(ious) >= iou_min else -1)
    d["yolo"] = yolo
    m = d[d.yolo >= 0]
    sama = m.yolo == m.pohon
    ringkas = pd.DataFrame({
        "nilai": [len(d), len(m) / len(d), (m.yolo == m.kelas).mean(), (m.pohon == m.kelas).mean(), sama.mean(),
                  (m[sama].yolo == m[sama].kelas).mean() if sama.any() else np.nan,
                  (m[~sama].yolo == m[~sama].kelas).mean() if (~sama).any() else np.nan]},
        index=["objek test", "terdeteksi YOLO (IoU>=%.1f)" % iou_min, "akurasi YOLO (yang terdeteksi)", "akurasi pohon (objek yang sama)",
               "YOLO & pohon sepakat", "akurasi saat sepakat", "akurasi YOLO saat TIDAK sepakat"]).round(3)
    return ringkas

display(banding_yolo_vs_pohon())

In [ ]:
def nilai_kematangan(path, model_path=None, hari_ini=None):
    hari_ini = hari_ini or date.today()
    det = YOLO(str(model_path or PRED_MODEL))
    raw = np.asarray(_load640(path))
    r = det.predict(Image.fromarray(raw), imgsz=IMGSZ, conf=PRED_CONF, device=0 if torch.cuda.is_available() else "cpu", verbose=False)[0]
    im, rows = normalisasi_cahaya(raw, MODE_TERPILIH), []
    for i, b in enumerate(r.boxes):
        x1, y1, x2, y2 = [int(v) for v in b.xyxy[0].tolist()]
        if x2 - x1 < 8 or y2 - y1 < 8:
            continue
        f = pd.DataFrame([fitur_crop(im[y1:y2, x1:x2])])[kol_final]
        nm, ny = NAMES["fruit"][int(clf_final.predict(f)[0])], det.names[int(b.cls[0])]
        lo, hi = STAGE_DAYS.get(nm, (None, None))
        rows.append(dict(buah=i + 1, tahap_yolo=ny, tahap_fitur=nm, sepakat=(ny == nm), sisa_hari=f"{lo}-{hi}",
                         matang_paling_cepat=None if lo is None else hari_ini + timedelta(days=lo)))
    return pd.DataFrame(rows)

for _p in sorted((WORK / "data_fruit" / "test" / "images").glob("*.jpg"))[:3]:
    print(_p.name)
    display(nilai_kematangan(_p))

## 6. Ekspor TFLite 

In [ ]:
if RUN_EXPORT:
    for task in TRAIN_TASKS:
        for variant in ("baseline", "cbam"):
            best = RUNS / f"{task}_{variant}_s0" / "weights" / "best.pt"
            if not best.exists():
                print("best.pt tidak ada (hasil dipulihkan tanpa bobot?):", best.parent.parent.name)
                continue
            for tag, kw in {"float32": {}, "fp16": {"half": True}}.items():
                try:
                    out = YOLO(str(best)).export(format="tflite", imgsz=IMGSZ, **kw)
                    print(task, variant, tag, "->", out, f"{Path(out).stat().st_size / 1e6:.2f} MB" if Path(out).is_file() else "")
                except Exception as e:
                    print("ekspor gagal:", task, variant, tag, "|", repr(e)[:200])
else:
    print("RUN_EXPORT=False: bagian ini dilewati.")

## 7. Simpan hasil

In [ ]:
shutil.make_archive("/kaggle/working/hasil_runs", "zip", str(RUNS))

final = Path("/kaggle/working/model_final")
final.mkdir(exist_ok=True)
for _t in TRAIN_TASKS:
    for _v in ("baseline", "cbam"):
        _b = RUNS / f"{_t}_{_v}_s0" / "weights" / "best.pt"
        if _b.exists():
            shutil.copy2(_b, final / f"{_t}_{_v}_s0_best.pt")
shutil.copy2("/kaggle/working/cbam_module.py", final / "cbam_module.py")
print("model_final:", sorted(os.listdir(final)))
print("working:", sorted(os.listdir("/kaggle/working")))